# C_1: Static Embeddings (FastText, pretrained, versi subword)

Data: B_2 (artikel Inggris terfilter). Split: sama dengan B_2.

Cara kerja:
1. Tiap hari: teks diubah menjadi kata-kata huruf kecil.
2. Tiap kata dicari vektornya di FastText `cc.en.300` (300 angka per kata). Model `.bin` menyimpan vektor
   untuk potongan huruf (character n-gram), jadi kata yang tidak ada di kosakata tetap mendapat vektor
   dari potongan hurufnya.
3. Vektor semua kata di satu hari dirata-rata menjadi 1 vektor harian (300 angka).
4. PCA mengecilkan 300 angka menjadi `N_COMP` angka. PCA hanya dilatih di hari train (supaya tidak bocor ke val/test).
5. Fitur ini digabung dengan fitur harga, lalu dinilai di validation lewat `c_common`.

Catatan: versi pertama C_1 memakai file `.vec` (hanya lookup kata utuh). Versi ini memakai `.bin` (subword).
Karena itu `n_configs_tried=2` di sel evaluasi.


In [1]:
from c_common import *
import re, urllib.request
from sklearn.decomposition import PCA

D, text = load_all()   # B_2: hanya artikel Inggris terfilter
print({s: len(d) for s, d in D.items()}, text.shape)

{'train': 825, 'val': 176, 'test': 178} (1202, 5)


## Pengaturan (ubah baris-baris ini saja)

In [2]:
SOURCE = "konten_gabungan"   # teks yang dipakai: "judul_gabungan" | "lead_paragraph_gabungan" | "konten_gabungan"
N_COMP = 20                  # jumlah komponen PCA (fitur teks akhir)
DOWNLOAD = False             # ganti True SEKALI untuk mengunduh FastText .bin (+/- 4,5 GB) lalu di-unzip (+/- 7,2 GB). Butuh ruang kosong di D:

VEC_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz"
# Disimpan di luar repo (drive C: hampir penuh, dan file besar tidak boleh di-commit). Ganti baris ini kalau path-mu beda.
VEC_GZ = Path(r"D:\nlp_embeddings\cc.en.300.bin.gz")   # file unduhan (terkompresi)
VEC_PATH = Path(r"D:\nlp_embeddings\cc.en.300.bin")      # hasil unzip (+/- 7,2 GB); ini yang dibaca gensim

In [3]:
import gzip, shutil

if DOWNLOAD and not VEC_PATH.exists():
    VEC_GZ.parent.mkdir(parents=True, exist_ok=True)
    if not VEC_GZ.exists():
        urllib.request.urlretrieve(VEC_URL, VEC_GZ)
        print("Unduh selesai:", VEC_GZ, round(VEC_GZ.stat().st_size / 1e9, 2), "GB")
    # gensim error (AssertionError) kalau membaca .bin.gz langsung, jadi di-unzip dulu
    with gzip.open(VEC_GZ, "rb") as fi, open(VEC_PATH, "wb") as fo:
        shutil.copyfileobj(fi, fo, 64 << 20)
    print("Unzip selesai:", VEC_PATH, round(VEC_PATH.stat().st_size / 1e9, 2), "GB")
assert VEC_PATH.exists(), "File FastText belum ada. Set DOWNLOAD = True di sel pengaturan lalu jalankan ulang."


## 1. Buka model FastText (hemat memori)
File `.bin` berukuran 7,2 GB. `gensim.load_facebook_vectors` memuat semuanya ke RAM (+/- 7 GB) dan di Windows
gagal membacanya (`AssertionError: expected ..., got ...`). Jadi di sini tabel vektornya dibuka dengan
`np.memmap` (dibaca dari disk hanya baris yang dibutuhkan) dan vektor tiap kata dihitung sendiri:
- kata yang ada di kosakata: rata-rata vektor kata itu dan vektor potongan hurufnya (character n-gram)
- kata yang tidak ada di kosakata: rata-rata vektor potongan hurufnya saja

Ini sama dengan cara FastText/gensim menghitungnya. Sudah dicek: untuk kata di kosakata, hasilnya sama dengan vektor di file `.vec`.
Butuh `pip install gensim` (dipakai untuk membaca kosakata dan menghitung hash potongan huruf).

In [4]:
import gensim.models._fasttext_bin as fb
from gensim.models.fasttext import ft_ngram_hashes


def _memmap_matrix(fin, new_format=True):
    """Ganti pembaca matriks bawaan gensim: jangan muat ke RAM, cukup petakan file (memmap)."""
    if new_format:
        fb._struct_unpack(fin, "@?")
    n, dim = fb._struct_unpack(fin, "@2q")
    pos = fin.tell()
    mm = np.memmap(VEC_PATH, dtype=np.float32, mode="r", offset=pos, shape=(n, dim))
    fin.seek(pos + n * dim * 4)
    return mm


fb._load_matrix = _memmap_matrix
with open(VEC_PATH, "rb") as fin:
    FT = fb.load(fin, encoding="utf-8", full_model=False)   # hanya kosakata + parameter; matriks lewat memmap

WORDS = {w: i for i, w in enumerate(FT.raw_vocab)}          # kata -> nomor baris
NW = len(WORDS)
MM = FT.vectors_ngrams                                       # baris 0..NW-1 = kata, baris sesudahnya = potongan huruf
DIM = MM.shape[1]
print(NW, "kata di kosakata,", DIM, "dimensi, potongan huruf", FT.minn, "-", FT.maxn, ",", FT.bucket, "bucket")


def word_vector(w):
    """Vektor 1 kata: rata-rata vektor potongan hurufnya (+ vektor kata itu sendiri kalau ada di kosakata)."""
    rows = [NW + h for h in ft_ngram_hashes(w, FT.minn, FT.maxn, FT.bucket)]
    if w in WORDS:
        rows.append(WORDS[w])
    return MM[rows].mean(axis=0) if rows else np.zeros(DIM, dtype=np.float32)   # kata sangat pendek & tidak dikenal -> nol

2000000 kata di kosakata, 300 dimensi, potongan huruf 5 - 5 , 2000000 bucket


## 2. Vektor harian (rata-rata vektor kata)
Tiap kata unik dihitung vektornya sekali, lalu dirata-rata per hari. Hari tanpa teks diberi vektor nol.

In [5]:
TOKEN = re.compile(r"[a-z]+")

docs = [TOKEN.findall(str(t).lower()) for t in text[SOURCE].fillna("")]
vocab = sorted({w for d in docs for w in d})
WV = {w: i for i, w in enumerate(vocab)}
MAT = np.vstack([word_vector(w) for w in vocab])      # vektor tiap kata unik
print(len(vocab), "kata unik; di luar kosakata FastText:", sum(w not in WORDS for w in vocab))

daily = np.vstack([MAT[[WV[w] for w in d]].mean(axis=0) if d else np.zeros(DIM, dtype=np.float32) for d in docs])
has_text = (np.abs(daily).sum(axis=1) > 0)
print(daily.shape, "| hari dengan teks:", int(has_text.sum()), "dari", len(daily))

60669 kata unik; di luar kosakata FastText: 14637
(1202, 300) | hari dengan teks: 1182 dari 1202


## 3. PCA (dilatih di train saja) dan simpan fitur

In [6]:
days = text.index
is_train = days.isin(D["train"]["date"]) & has_text      # hanya hari train yang punya teks
pca = PCA(n_components=N_COMP, random_state=SEED).fit(daily[is_train])
print("variansi dijelaskan:", round(pca.explained_variance_ratio_.sum(), 3))

emb_cols = [f"ft_{SOURCE[:5]}_pc{i:02d}" for i in range(N_COMP)]
fitur = pd.DataFrame(pca.transform(daily), index=days, columns=emb_cols)
fitur.loc[~has_text] = 0.0                                # hari tanpa teks -> 0
fitur = fitur.reset_index()                               # kolom 'date' + fitur

FEATURES_C.mkdir(parents=True, exist_ok=True)
fitur.to_csv(FEATURES_C / "C1_fasttext.csv", index=False)  # simpan agar tidak perlu hitung ulang
fitur.shape

variansi dijelaskan: 0.778


(1202, 21)

## 4. Nilai di validation
Test **tidak** disentuh di sini. Pilih varian terbaik dulu di validation (bersama varian C lain), baru `final_test` sekali.

In [7]:
for s in D:
    D[s] = D[s].merge(fitur, on="date", how="left").fillna(0)

evaluate_val("C1_LR",  "LR",  D, PRICE + emb_cols, n_configs_tried=2, note=f"FastText cc.en.300 (bin, subword), {SOURCE}, PCA{N_COMP}")
evaluate_val("C1_HGB", "HGB", D, PRICE + emb_cols, n_configs_tried=2, note=f"FastText cc.en.300 (bin, subword), {SOURCE}, PCA{N_COMP}")

C1_LR [LR] val: DA=0.500, BalAcc=0.501, F1_macro=0.500, MCC=0.001, AUC=0.441
C1_HGB [HGB] val: DA=0.528, BalAcc=0.533, F1_macro=0.523, MCC=0.068, AUC=0.547


{'DA': 0.5284090909090909,
 'BalAcc': 0.5327084680025856,
 'F1_macro': 0.5228482769884044,
 'MCC': 0.06752304944551178,
 'AUC': 0.5468648998060763}

## 5. Bandingkan dengan hasil B_2
Hasil B_2 dihitung ulang dari `data/results_filtered/predictions.csv` (Tugas 2). Baris C_1 hanya punya skor validation; test belum disentuh.

In [8]:
import json

pred = pd.read_csv(DATA / "results_filtered" / "predictions.csv", parse_dates=["date"])
rows = []
for name in ["Naive: Majority", "LR [Harga]", "HGB [Harga]", "LR [NLP]", "LR [Harga+NLP]", "HGB [Harga+NLP]"]:
    for s in ["val", "test"]:
        d = pred[pred["split"] == s]
        sc = d.get("score::" + name)
        rows.append({"model": "B_2 " + name, "split": s, **metrics(d["target"], d["pred::" + name], sc)})
for name in ["C1_LR", "C1_HGB"]:   # C_1: hanya validation
    rows.append({"model": name, "split": "val", **json.loads((RESULTS_C / f"{name}.json").read_text())["val"]})
pd.DataFrame(rows).round(3)

,model,split,DA,BalAcc,F1_macro,MCC,AUC
0,B_2 Naive: Majority,val,0.517,0.500,0.341,0.000,NaN
1,B_2 Naive: Majority,test,0.579,0.500,0.367,0.000,NaN
2,B_2 LR [Harga],val,0.506,0.508,0.504,0.016,0.501
3,B_2 LR [Harga],test,0.517,0.515,0.513,0.031,0.527
4,B_2 HGB [Harga],val,0.523,0.521,0.520,0.042,0.503
5,B_2 HGB [Harga],test,0.500,0.503,0.498,0.005,0.538
6,B_2 LR [NLP],val,0.545,0.546,0.545,0.093,0.544
7,B_2 LR [NLP],test,0.539,0.544,0.538,0.087,0.563
8,B_2 LR [Harga+NLP],val,0.528,0.531,0.527,0.062,0.529
9,B_2 LR [Harga+NLP],test,0.500,0.505,0.499,0.009,0.561
